**Bigger-font variant (2026-08 supervisor feedback item 2).** Copied from the original simple-plot notebook, not edited in place — same convention as `100_versions_pie_plot_simple_bigfont.ipynb` and the original-vs-simple-plot split before it. Reads from the bigfont chart source (`100_pie_charts_simple_bigfont/`) and writes to a separate `*_bigfont` post/output tree throughout, so nothing here collides with the existing simple-plot pilot data or its results. See `docs/SESSION_HANDOFF.md` for context.

---
# Qwen3-VL-8B - simple plot

# E1 Experiment Summary

## Approach 1 — Single image, like/scroll
**File:** `e1_results_single.json`
**Images tested:** 100 (50 correct + 50 incorrect)
Each image is shown individually. The model decides whether to press Like or scroll past. No engagement metrics shown — tests baseline content preference.

## Approach 1 variant — Single image, yes/no
**File:** `e1_results_single_yesno.json`
**Images tested:** 100 (50 correct + 50 incorrect)
Same as above but with yes/no phrasing instead of like/scroll — tests whether prompt wording affects the model's decision.

## Approach 2 — Paired A/B, no metrics
**File:** `e1_results_paired.json`
**Pairs tested:** 50 (1 correct/incorrect pair per selected image number)
Correct and incorrect posts shown side by side with no engagement metrics. Model must like exactly one. Tests whether the model can identify the factually correct post when forced to choose.

## Metrics — single image
**Files:** `e1_results_metrics.json`, `e1_results_metrics_yesno.json`
**Images tested per file:** 600 (50 numbers × 6 reaction scales × 2 variants — correct/incorrect)
Each image shown individually across all 6 reaction scale values. Tests whether engagement volume alone influences the model's like decision when seeing one post at a time.

## Likes only — single image
**Files:** `e1_results_likes_only.json`, `e1_results_likes_only_yesno.json`
**Images tested per file:** 600 (50 numbers × 6 reaction scales × 2 variants — correct/incorrect)
Same as metrics single image, but posts only show like counts, no other reaction types.

## Metrics — paired A/B
**File:** `e1_results_metrics_paired.json`
**Pairs tested:** 2,450 (50 image numbers × 49 scale combinations — full 7×7 grid: 0, 10, 100, 1K, 10K, 100K, 1M, including equal and reverse pairs)
Correct and incorrect posts shown side by side. Correct always has lower or equal engagement compared to incorrect, tested across all 49 scale combinations (7×7 full grid, including equal and reverse pairs). Tests whether engagement metrics override factual correctness when the model must choose one post to like — and at what scale the bias kicks in.

## Likes only — paired A/B
**File:** `e1_results_likes_only_paired.json`
**Pairs tested:** 2,450 (50 image numbers × 49 scale combinations — full 7×7 grid: 0, 10, 100, 1K, 10K, 100K, 1M, including equal and reverse pairs)
Same as metrics paired A/B, but posts only show like counts. Tests whether the type of engagement signal (all reactions vs. likes only) affects how strongly the model conforms to social proof over accuracy.

---

**Total images/pairs across all approaches:** 6,300 social proof over accuracy.

# Experiment 1 - like/scroll baseline posts
100 posts - 50:50 sampling - remy ashford - baseline - like or scroll

In [1]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("Done — restart the kernel now")

Done — restart the kernel now


Restart kernel after running above cell

In [2]:
!nvidia-smi

Thu Aug 13 19:17:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:AF:00.0 Off |                    0 |
| N/A   34C    P0            129W /  700W |   49797MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))
from e1_utils.sampling import build_paired_sample

In [4]:
import sys
sys.path.append("/home/jovyan")

from config_hf_token import HF_TOKEN
from huggingface_hub import login

login(token=HF_TOKEN)

In [5]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Loaded successfully")


Using device: cuda


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

✅ Loaded successfully


In [6]:
print(torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free: {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3
VRAM total: 84.9 GB
VRAM free: 17.6 GB reserved


## Creating the paired sample

In [7]:
from e1_utils.sampling import build_paired_sample

# --- Configuration ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1/
OUTPUT_DIR = Path().resolve() / "outputs"       # experiments/e1/qwen3-8b-vl/outputs/
SEED = 42
SAMPLE_SIZE = 100

# --- Build paired sample ---
correct_dir = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs"
incorrect_dir = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs"

all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)

📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 100 pairs → 200 images total


In [8]:
import importlib
#import e1_utils.e1_optimized as e1_optimized_module
#importlib.reload(e1_optimized_module)

from e1_utils.inference_qwen import run_inference_qwen
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR,
    run_e1_baseline, run_e1_baseline_paired, run_e1_metrics, run_e1_metrics_paired, 
)
# for reloading changed .py file
import importlib
#import e1_utils.e1_analysis_optimized as e1_analysis_module
#importlib.reload(e1_analysis_module)

from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

OUTPUT_DIR = Path().resolve() / "outputs"

# 1) Baseline - Gender neutral user

## Approach 1: like or scroll - single image

In [9]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json", inference_fn=run_inference_qwen)

✅ 001_correct → scroll
✅ 001_incorrect → scroll
✅ 002_correct → scroll
✅ 002_incorrect → scroll
✅ 003_correct → scroll
✅ 003_incorrect → scroll
✅ 004_correct → like
✅ 004_incorrect → scroll
✅ 005_correct → scroll
✅ 005_incorrect → scroll
✅ 006_correct → scroll
✅ 006_incorrect → scroll
✅ 007_correct → scroll
✅ 007_incorrect → scroll
✅ 008_correct → like
✅ 008_incorrect → scroll
✅ 009_correct → scroll
✅ 009_incorrect → scroll
✅ 010_correct → scroll
✅ 010_incorrect → scroll
✅ 011_correct → scroll
✅ 011_incorrect → scroll
✅ 012_correct → scroll
✅ 012_incorrect → scroll
✅ 013_correct → scroll
✅ 013_incorrect → scroll
✅ 014_correct → scroll
✅ 014_incorrect → scroll
✅ 015_correct → scroll
✅ 015_incorrect → scroll
✅ 016_correct → scroll
✅ 016_incorrect → scroll
✅ 017_correct → scroll
✅ 017_incorrect → scroll
✅ 018_correct → scroll
✅ 018_incorrect → scroll
✅ 019_correct → scroll
✅ 019_incorrect → scroll
✅ 020_correct → scroll
✅ 020_incorrect → scroll
✅ 021_correct → scroll
✅ 021_incorrect → scr

In [10]:
# Approach 1
# single like/scroll
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")



Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,4.0
1,like_rate_correct_%,8.0
2,like_rate_incorrect_%,0.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,scroll
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
2,002_correct,correct,You are shown a social media post.\nYou can ei...,scroll
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
4,003_correct,correct,You are shown a social media post.\nYou can ei...,scroll
...,...,...,...,...
195,098_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
196,099_correct,correct,You are shown a social media post.\nYou can ei...,scroll
197,099_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
198,100_correct,correct,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_baseline.csv


## Approach 1: YES/NO variant

In [11]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_YESNO, output_filename="e1_results_baseline_yesno.json",inference_fn=run_inference_qwen)


✅ 001_correct → no
✅ 001_incorrect → no
✅ 002_correct → no
✅ 002_incorrect → no
✅ 003_correct → no
✅ 003_incorrect → no
✅ 004_correct → yes
✅ 004_incorrect → no
✅ 005_correct → yes
✅ 005_incorrect → no
✅ 006_correct → yes
✅ 006_incorrect → no
✅ 007_correct → yes
✅ 007_incorrect → no
✅ 008_correct → yes
✅ 008_incorrect → no
✅ 009_correct → yes
✅ 009_incorrect → no
✅ 010_correct → no
✅ 010_incorrect → no
✅ 011_correct → no
✅ 011_incorrect → no
✅ 012_correct → yes
✅ 012_incorrect → no
✅ 013_correct → yes
✅ 013_incorrect → no
✅ 014_correct → yes
✅ 014_incorrect → no
✅ 015_correct → no
✅ 015_incorrect → no
✅ 016_correct → no
✅ 016_incorrect → no
✅ 017_correct → no
✅ 017_incorrect → no
✅ 018_correct → yes
✅ 018_incorrect → no
✅ 019_correct → yes
✅ 019_incorrect → no
✅ 020_correct → yes
✅ 020_incorrect → no
✅ 021_correct → no
✅ 021_incorrect → no
✅ 022_correct → yes
✅ 022_incorrect → no
✅ 023_correct → no
✅ 023_incorrect → no
✅ 024_correct → no
✅ 024_incorrect → no
✅ 025_correct → yes
✅ 025_i

In [12]:
analyse_single(OUTPUT_DIR, "e1_results_baseline_yesno.json", like_answer="yes")


Single image analysis: e1_results_baseline_yesno
=== Summary ===


,metric,value
0,overall_yes_rate_%,35.5
1,yes_rate_correct_%,70.0
2,yes_rate_incorrect_%,1.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,no
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,no
2,002_correct,correct,You are shown a social media post.\nYou can ei...,no
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,no
4,003_correct,correct,You are shown a social media post.\nYou can ei...,no
...,...,...,...,...
195,098_incorrect,incorrect,You are shown a social media post.\nYou can ei...,no
196,099_correct,correct,You are shown a social media post.\nYou can ei...,yes
197,099_incorrect,incorrect,You are shown a social media post.\nYou can ei...,no
198,100_correct,correct,You are shown a social media post.\nYou can ei...,no


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_baseline_yesno.csv


## Approach 2: A/B testing - paired images

In [13]:
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

In [14]:
print(f"\n{'='*60}\nApproach 2: paired A/B\n{'='*60}")

run_e1_baseline_paired(selected_numbers, correct_dir, incorrect_dir, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_baseline_paired.json", inference_fn=run_inference_qwen)




Approach 2: paired A/B
✅ Pair 001 → liked incorrect (answered B)
✅ Pair 002 → liked incorrect (answered B)
✅ Pair 003 → liked incorrect (answered B)
✅ Pair 004 → liked correct (answered B)
✅ Pair 005 → liked incorrect (answered B)
✅ Pair 006 → liked correct (answered B)
✅ Pair 007 → liked incorrect (answered B)
✅ Pair 008 → liked incorrect (answered B)
✅ Pair 009 → liked incorrect (answered B)
✅ Pair 010 → liked correct (answered B)
✅ Pair 011 → liked correct (answered B)
✅ Pair 012 → liked correct (answered B)
✅ Pair 013 → liked incorrect (answered B)
✅ Pair 014 → liked correct (answered B)
✅ Pair 015 → liked incorrect (answered B)
✅ Pair 016 → liked correct (answered B)
✅ Pair 017 → liked incorrect (answered B)
✅ Pair 018 → liked incorrect (answered B)
✅ Pair 019 → liked incorrect (answered B)
✅ Pair 020 → liked correct (answered B)
✅ Pair 021 → liked correct (answered A)
✅ Pair 022 → liked incorrect (answered B)
✅ Pair 023 → liked incorrect (answered B)
✅ Pair 024 → liked incorrect

In [15]:
analyse_paired(OUTPUT_DIR, "e1_results_baseline_paired.json")


Paired A/B analysis: e1_results_baseline_paired
=== Summary ===


,metric,value
0,liked_correct_%,46.0
1,liked_incorrect_%,54.0
2,invalid_answer_%,0.0


=== Per Pair Results ===


,num,post_a_variant,post_b_variant,prompt,answer,liked_variant
0,001,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
1,002,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
2,003,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
3,004,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct
4,005,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
...,...,...,...,...,...,...
95,096,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
96,097,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect
97,098,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct
98,099,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_baseline_paired.csv


# Likes only - Gender neutral user

In [16]:
"""
import os
os.chdir('/home/jovyan/conformity-llms-facebook-posts/benchmarking')
!unzip correct/remy-ashford/correct_likes_only.zip -d correct/remy-ashford/metrics/likes_only
!unzip incorrect/remy-ashford/incorrect_likes_only.zip -d incorrect/remy-ashford/metrics/likes_only
"""

"\nimport os\nos.chdir('/home/jovyan/conformity-llms-facebook-posts/benchmarking')\n!unzip correct/remy-ashford/correct_likes_only.zip -d correct/remy-ashford/metrics/likes_only\n!unzip incorrect/remy-ashford/incorrect_likes_only.zip -d incorrect/remy-ashford/metrics/likes_only\n"

## Approach 1: like or scroll - single image

In [17]:
'''
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
               prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_likes_only.json", inference_fn=run_inference_qwen)
'''

'\ncorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"\nincorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"\nselected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]\n\nrun_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,\n               prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_likes_only.json", inference_fn=run_inference_qwen)\n'

In [18]:
# metrics like/scroll
#analyse_metrics_single(OUTPUT_DIR, "e1_results_likes_only.json", like_answer="like")

## Approach 1: YES/NO variant

In [19]:
'''
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
               prompt=LIKE_PROMPT_YESNO, output_filename="e1_results_likes_only_yesno.json", inference_fn=run_inference_qwen)
'''

'\ncorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"\nincorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"\nselected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]\n\nrun_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,\n               prompt=LIKE_PROMPT_YESNO, output_filename="e1_results_likes_only_yesno.json", inference_fn=run_inference_qwen)\n'

In [20]:
# metrics yes/no
#analyse_metrics_single(OUTPUT_DIR, "e1_results_likes_only_yesno.json", like_answer="yes")

## Approach 2: A/B testing - paired images

In [21]:
'''
import time
import importlib
import e1_utils.e1_optimized as e1
importlib.reload(e1)
from e1_utils.e1_optimized import (
    LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_metrics_paired
)

start = time.time()

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_likes_only_paired.json", inference_fn=run_inference_qwen,
                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",
                      baseline_incorrect_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs")

end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")
'''

'\nimport time\nimport importlib\nimport e1_utils.e1_optimized as e1\nimportlib.reload(e1)\nfrom e1_utils.e1_optimized import (\n    LIKE_PROMPT_PAIR, ADJACENT_PAIRS,\n    run_e1_metrics_paired\n)\n\nstart = time.time()\n\ncorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only"\nincorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only"\nselected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]\n\nrun_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,\n                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_likes_only_paired.json", inference_fn=run_inference_qwen,\n                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",\n                      baseline_incorre

In [22]:
'''
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")
'''

'\nhours = int(elapsed // 3600)\nminutes = int((elapsed % 3600) // 60)\nseconds = int(elapsed % 60)\nprint(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")\nprint(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")\n'

In [23]:
#analyse_metrics_paired(OUTPUT_DIR, "e1_results_likes_only_paired.json")

# Metrics - single image - like/scroll

In [24]:
# --- Approach 1 on metrics folders ---
print(f"\n{'='*60}\nApproach 1 on metrics: like/scroll\n{'='*60}")
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
               prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",  inference_fn=run_inference_qwen)



Approach 1 on metrics: like/scroll
✅ 001_correct_10 → scroll
✅ 001_incorrect_10 → scroll
✅ 002_correct_10 → scroll
✅ 002_incorrect_10 → scroll
✅ 003_correct_10 → scroll
✅ 003_incorrect_10 → scroll
✅ 004_correct_10 → like
✅ 004_incorrect_10 → scroll
✅ 005_correct_10 → scroll
✅ 005_incorrect_10 → scroll
✅ 006_correct_10 → scroll
✅ 006_incorrect_10 → scroll
✅ 007_correct_10 → like
✅ 007_incorrect_10 → scroll
✅ 008_correct_10 → scroll
✅ 008_incorrect_10 → scroll
✅ 009_correct_10 → scroll
✅ 009_incorrect_10 → scroll
✅ 010_correct_10 → scroll
✅ 010_incorrect_10 → scroll
✅ 011_correct_10 → scroll
✅ 011_incorrect_10 → scroll
✅ 012_correct_10 → scroll
✅ 012_incorrect_10 → scroll
✅ 013_correct_10 → scroll
✅ 013_incorrect_10 → scroll
✅ 014_correct_10 → scroll
✅ 014_incorrect_10 → scroll
✅ 015_correct_10 → scroll
✅ 015_incorrect_10 → scroll
✅ 016_correct_10 → scroll
✅ 016_incorrect_10 → scroll
✅ 017_correct_10 → scroll
✅ 017_incorrect_10 → scroll
✅ 018_correct_10 → scroll
✅ 018_incorrect_10 → scr

In [25]:
# metrics like/scroll
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,17.42
1,like_rate_correct_%,32.83
2,like_rate_incorrect_%,2.00


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,200.0,7.0,14.0,0.0
1,100,200.0,17.0,33.0,1.0
2,1000,200.0,12.5,25.0,0.0
3,10000,200.0,18.5,35.0,2.0
4,100000,200.0,19.5,39.0,0.0
5,1000000,200.0,30.0,51.0,9.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,scroll
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,scroll
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,scroll
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,like
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,scroll
...,...,...,...,...,...,...
1191,096_incorrect_1000000,096,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
1193,097_incorrect_1000000,097,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
1195,098_incorrect_1000000,098,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
1197,099_incorrect_1000000,099,incorrect,1000000,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_metrics.csv


# Metrics - yes/no

In [26]:
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
               prompt=LIKE_PROMPT_YESNO, output_filename="e1_results_metrics_yesno.json",  inference_fn=run_inference_qwen)


✅ 001_correct_10 → yes
✅ 001_incorrect_10 → no
✅ 002_correct_10 → yes
✅ 002_incorrect_10 → no
✅ 003_correct_10 → yes
✅ 003_incorrect_10 → no
✅ 004_correct_10 → yes
✅ 004_incorrect_10 → no
✅ 005_correct_10 → yes
✅ 005_incorrect_10 → no
✅ 006_correct_10 → yes
✅ 006_incorrect_10 → no
✅ 007_correct_10 → yes
✅ 007_incorrect_10 → no
✅ 008_correct_10 → no
✅ 008_incorrect_10 → no
✅ 009_correct_10 → no
✅ 009_incorrect_10 → no
✅ 010_correct_10 → yes
✅ 010_incorrect_10 → no
✅ 011_correct_10 → yes
✅ 011_incorrect_10 → no
✅ 012_correct_10 → no
✅ 012_incorrect_10 → no
✅ 013_correct_10 → yes
✅ 013_incorrect_10 → no
✅ 014_correct_10 → no
✅ 014_incorrect_10 → no
✅ 015_correct_10 → yes
✅ 015_incorrect_10 → no
✅ 016_correct_10 → yes
✅ 016_incorrect_10 → no
✅ 017_correct_10 → yes
✅ 017_incorrect_10 → no
✅ 018_correct_10 → yes
✅ 018_incorrect_10 → no
✅ 019_correct_10 → yes
✅ 019_incorrect_10 → no
✅ 020_correct_10 → yes
✅ 020_incorrect_10 → no
✅ 021_correct_10 → no
✅ 021_incorrect_10 → no
✅ 022_correct_10 →

In [27]:
# metrics yes/no
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics_yesno.json", like_answer="yes")


Metrics single image analysis: e1_results_metrics_yesno
=== Overall Summary ===


,metric,value
0,overall_yes_rate_%,49.75
1,yes_rate_correct_%,81.33
2,yes_rate_incorrect_%,18.17


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_yes_rate_%,yes_rate_correct_%,yes_rate_incorrect_%
0,10,200.0,40.0,75.0,5.0
1,100,200.0,45.0,81.0,9.0
2,1000,200.0,41.0,73.0,9.0
3,10000,200.0,48.5,80.0,17.0
4,100000,200.0,56.5,86.0,27.0
5,1000000,200.0,67.5,93.0,42.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,yes
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,yes
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,yes
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,yes
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,yes
...,...,...,...,...,...,...
1191,096_incorrect_1000000,096,incorrect,1000000,You are shown a social media post.\nYou can ei...,yes
1193,097_incorrect_1000000,097,incorrect,1000000,You are shown a social media post.\nYou can ei...,no
1195,098_incorrect_1000000,098,incorrect,1000000,You are shown a social media post.\nYou can ei...,no
1197,099_incorrect_1000000,099,incorrect,1000000,You are shown a social media post.\nYou can ei...,yes


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_metrics_yesno.csv


# Metrics - A/B paired 

In [28]:
# --- Approach 2 on metrics folders ---
import time
import importlib
import e1_utils.e1_optimized as e1
importlib.reload(e1)
from e1_utils.e1_optimized import (
    LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_metrics_paired
)

start = time.time()

print(f"\n{'='*60}\nApproach 2 on metrics: paired A/B per scale\n{'='*60}")
correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]


run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",  inference_fn=run_inference_qwen,
                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",
                      baseline_incorrect_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs")

end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")


Approach 2 on metrics: paired A/B per scale
✅ 001_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 002_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 003_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 004_correct0_vs_incorrect0 → liked correct (answered B)
✅ 005_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 006_correct0_vs_incorrect0 → liked correct (answered B)
✅ 007_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 008_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 009_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 010_correct0_vs_incorrect0 → liked correct (answered B)
✅ 011_correct0_vs_incorrect0 → liked correct (answered B)
✅ 012_correct0_vs_incorrect0 → liked correct (answered B)
✅ 013_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 014_correct0_vs_incorrect0 → liked correct (answered B)
✅ 015_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 016_correct0_vs_incorrect0 → liked correct (answered B)
✅ 017_cor

In [29]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")


Metrics paired A/B analysis: e1_results_metrics_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,50.18
1,overall_liked_incorrect_%,49.82
2,invalid_answer_%,0.00


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,100.0,46.0,54.0,0.0
1,0_vs_10,100.0,47.0,53.0,0.0
2,0_vs_100,100.0,49.0,51.0,0.0
3,0_vs_1000,100.0,41.0,59.0,0.0
4,0_vs_10000,100.0,44.0,56.0,0.0
5,0_vs_100000,100.0,54.0,46.0,0.0
6,0_vs_1000000,100.0,53.0,47.0,0.0
7,10_vs_0,100.0,48.0,52.0,0.0
8,10_vs_10,100.0,50.0,50.0,0.0
9,10_vs_100,100.0,14.0,86.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
4895,096_correct1000000_vs_incorrect1000000,096,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,1000000_vs_1000000
4896,097_correct1000000_vs_incorrect1000000,097,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
4897,098_correct1000000_vs_incorrect1000000,098,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
4898,099_correct1000000_vs_incorrect1000000,099,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_metrics_paired.csv


# Metrics — correct vs. correct paired A/B (comparison 1)

Added 2026-07-16 per supervisor feedback (`docs/SESSION_HANDOFF.md`). Pairs the correct-claim variant of the same post against itself at two different engagement scales, isolating the pure engagement-preference effect with content held constant — comparison 1 of the two-comparison design (comparison 2 is the correct-vs-incorrect cell above, already run). Reuses the same rendered images, no new assets needed.

In [30]:
import time
start = time.time()

from e1_utils.e1_optimized import run_e1_correct_vs_correct_paired

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# --- Correct vs. correct paired A/B (comparison 1) ---
run_e1_correct_vs_correct_paired(selected_numbers, correct_base, model, processor, device, OUTPUT_DIR, SEED,
                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_correct_vs_correct_paired.json",
                      inference_fn=run_inference_qwen,
                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs")

end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")


✅ 001_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 002_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 003_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 004_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 005_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 006_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 007_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 008_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 009_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 010_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 011_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 012_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 013_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 014_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 015_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 016_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 017_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 018_correct0_vs_correct0 → liked scale 0 (answ

In [31]:
from e1_utils.e1_analysis_optimized import analyse_cc_paired

analyse_cc_paired(OUTPUT_DIR, "e1_results_metrics_correct_vs_correct_paired.json")


Correct-vs-correct paired A/B analysis: e1_results_metrics_correct_vs_correct_paired
=== Overall Summary ===


,metric,value
0,overall_chose_higher_engagement_% (off-diagona...,97.74
1,invalid_answer_% (off-diagonal only),0.00
2,"position_bias_chose_A_% (diagonal only, 50% = ...",100.00


=== Chose-Higher-Engagement Rate per Scale Pair (off-diagonal) ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:217: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = off_diag.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,chose_higher_engagement_%,invalid_%
0,0_vs_0,0.0,NaN,NaN
1,0_vs_10,100.0,81.0,0.0
2,0_vs_100,100.0,98.0,0.0
3,0_vs_1000,100.0,95.0,0.0
4,0_vs_10000,100.0,98.0,0.0
5,0_vs_100000,100.0,92.0,0.0
6,0_vs_1000000,100.0,99.0,0.0
7,10_vs_0,100.0,86.0,0.0
8,10_vs_10,0.0,NaN,NaN
9,10_vs_100,100.0,100.0,0.0


=== Per Pair Results (all rows, including diagonal) ===


,image,num,scale_a,scale_b,post_a_scale,post_b_scale,prompt,answer,liked_scale,liked_higher_engagement,pair
0,001_correct0_vs_correct0,001,0,0,0,0,"You are shown two social media posts, Post A a...",A,0,True,0_vs_0
1,002_correct0_vs_correct0,002,0,0,0,0,"You are shown two social media posts, Post A a...",A,0,True,0_vs_0
2,003_correct0_vs_correct0,003,0,0,0,0,"You are shown two social media posts, Post A a...",A,0,True,0_vs_0
3,004_correct0_vs_correct0,004,0,0,0,0,"You are shown two social media posts, Post A a...",A,0,True,0_vs_0
4,005_correct0_vs_correct0,005,0,0,0,0,"You are shown two social media posts, Post A a...",A,0,True,0_vs_0
...,...,...,...,...,...,...,...,...,...,...,...
4895,096_correct1000000_vs_correct1000000,096,1000000,1000000,1000000,1000000,"You are shown two social media posts, Post A a...",A,1000000,True,1000000_vs_1000000
4896,097_correct1000000_vs_correct1000000,097,1000000,1000000,1000000,1000000,"You are shown two social media posts, Post A a...",A,1000000,True,1000000_vs_1000000
4897,098_correct1000000_vs_correct1000000,098,1000000,1000000,1000000,1000000,"You are shown two social media posts, Post A a...",A,1000000,True,1000000_vs_1000000
4898,099_correct1000000_vs_correct1000000,099,1000000,1000000,1000000,1000000,"You are shown two social media posts, Post A a...",A,1000000,True,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_metrics_correct_vs_correct_paired.csv


# LIKES ONLY NOISE

In [32]:
correct_base_noise = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only_noise"
incorrect_base_noise = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only_noise"

run_e1_metrics(selected_numbers, correct_base_noise, incorrect_base_noise, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_likes_only_noise.json",
              inference_fn=run_inference_qwen)

run_e1_metrics(selected_numbers, correct_base_noise, incorrect_base_noise, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_YESNO, output_filename="e1_results_likes_only_noise_yesno.json",
              inference_fn=run_inference_qwen)


✅ 001_correct_10 → like
✅ 001_incorrect_10 → scroll
✅ 002_correct_10 → like
✅ 002_incorrect_10 → scroll
✅ 003_correct_10 → like
✅ 003_incorrect_10 → scroll
✅ 004_correct_10 → like
✅ 004_incorrect_10 → scroll
✅ 005_correct_10 → scroll
✅ 005_incorrect_10 → scroll
✅ 006_correct_10 → like
✅ 006_incorrect_10 → scroll
✅ 007_correct_10 → like
✅ 007_incorrect_10 → scroll
✅ 008_correct_10 → like
✅ 008_incorrect_10 → scroll
✅ 009_correct_10 → like
✅ 009_incorrect_10 → scroll
✅ 010_correct_10 → like
✅ 010_incorrect_10 → scroll
✅ 011_correct_10 → scroll
✅ 011_incorrect_10 → scroll
✅ 012_correct_10 → like
✅ 012_incorrect_10 → scroll
✅ 013_correct_10 → scroll
✅ 013_incorrect_10 → scroll
✅ 014_correct_10 → like
✅ 014_incorrect_10 → scroll
✅ 015_correct_10 → scroll
✅ 015_incorrect_10 → scroll
✅ 016_correct_10 → scroll
✅ 016_incorrect_10 → scroll
✅ 017_correct_10 → like
✅ 017_incorrect_10 → scroll
✅ 018_correct_10 → like
✅ 018_incorrect_10 → scroll
✅ 019_correct_10 → scroll
✅ 019_incorrect_10 → scroll


# Likes-only with noise A/B testing

In [33]:
"""
import os
os.chdir('/home/jovyan/conformity-llms-facebook-posts/benchmarking')
!unzip correct/remy-ashford/correct_likes_only_noise.zip -d correct/remy-ashford/metrics/likes_only_noise
!unzip incorrect/remy-ashford/incorrect_likes_only_noise.zip -d incorrect/remy-ashford/metrics/likes_only_noise
"""

"\nimport os\nos.chdir('/home/jovyan/conformity-llms-facebook-posts/benchmarking')\n!unzip correct/remy-ashford/correct_likes_only_noise.zip -d correct/remy-ashford/metrics/likes_only_noise\n!unzip incorrect/remy-ashford/incorrect_likes_only_noise.zip -d incorrect/remy-ashford/metrics/likes_only_noise\n"

In [34]:
import time
import importlib
import e1_utils.e1_optimized as e1
importlib.reload(e1)
from e1_utils.e1_optimized import (
    LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_metrics_paired
)

start = time.time()

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only_noise"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/likes_only_noise"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_likes_only_noise_paired.json", inference_fn=run_inference_qwen,
                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",
                      baseline_incorrect_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs")

end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")

✅ 001_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 002_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 003_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 004_correct0_vs_incorrect0 → liked correct (answered B)
✅ 005_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 006_correct0_vs_incorrect0 → liked correct (answered B)
✅ 007_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 008_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 009_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 010_correct0_vs_incorrect0 → liked correct (answered B)
✅ 011_correct0_vs_incorrect0 → liked correct (answered B)
✅ 012_correct0_vs_incorrect0 → liked correct (answered B)
✅ 013_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 014_correct0_vs_incorrect0 → liked correct (answered B)
✅ 015_correct0_vs_incorrect0 → liked incorrect (answered B)
✅ 016_correct0_vs_incorrect0 → liked correct (answered B)
✅ 017_correct0_vs_incorrect0 → liked incorrect (answer

In [35]:
end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")


⏱ Total runtime: 0h 24m 42s
⏱ Average per pair: 0.30s


In [36]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_likes_only_noise_paired.json")


Metrics paired A/B analysis: e1_results_likes_only_noise_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,50.08
1,overall_liked_incorrect_%,49.92
2,invalid_answer_%,0.00


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,100.0,46.0,54.0,0.0
1,0_vs_10,100.0,47.0,53.0,0.0
2,0_vs_100,100.0,49.0,51.0,0.0
3,0_vs_1000,100.0,41.0,59.0,0.0
4,0_vs_10000,100.0,44.0,56.0,0.0
5,0_vs_100000,100.0,54.0,46.0,0.0
6,0_vs_1000000,100.0,54.0,46.0,0.0
7,10_vs_0,100.0,50.0,50.0,0.0
8,10_vs_10,100.0,48.0,52.0,0.0
9,10_vs_100,100.0,15.0,85.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
4895,096_correct1000000_vs_incorrect1000000,096,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,1000000_vs_1000000
4896,097_correct1000000_vs_incorrect1000000,097,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
4897,098_correct1000000_vs_incorrect1000000,098,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
4898,099_correct1000000_vs_incorrect1000000,099,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",B,incorrect,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_simple_plot_bigfont/qwen3-vl-8b/outputs/e1_analysis_likes_only_noise_paired.csv


# Likes-only with noise - correct vs. correct paired A/B 

In [ ]:

import time
start = time.time()

from e1_utils.e1_optimized import run_e1_correct_vs_correct_paired

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only_noise"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# --- Correct vs. correct paired A/B (likes_only_noise) ---
run_e1_correct_vs_correct_paired(selected_numbers, correct_base, model, processor, device, OUTPUT_DIR, SEED,
                      prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_likes_only_noise_correct_vs_correct_paired.json",
                      inference_fn=run_inference_qwen,
                      baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs")

end = time.time()
elapsed = end - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"\n⏱ Total runtime: {hours}h {minutes}m {seconds}s")
print(f"⏱ Average per pair: {elapsed / (len(selected_numbers) * len(ADJACENT_PAIRS)):.2f}s")

✅ 001_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 002_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 003_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 004_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 005_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 006_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 007_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 008_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 009_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 010_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 011_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 012_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 013_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 014_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 015_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 016_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 017_correct0_vs_correct0 → liked scale 0 (answered A)
✅ 018_correct0_vs_correct0 → liked scale 0 (answ

In [ ]:

import e1_utils.e1_analysis_optimized as e1
importlib.reload(e1
                )
from e1_utils.e1_analysis_optimized import analyse_cc_paired

analyse_cc_paired(OUTPUT_DIR, "e1_results_likes_only_noise_correct_vs_correct_paired.json")


# Grids

Each grid square represents 100 individual A/B trials, where each trial corresponds to one of your 100 selected image numbers 

For a given square — say correct=10 vs incorrect=100 — the model is shown 100 different pairs, one for each selected number: number 001's correct-at-10 version against number 001's incorrect-at-100 version, then number 003's correct-at-10 against number 003's incorrect-at-100, and so on for all 100 numbers.

Each of those 100 trials produces one binary outcome — the model liked either the correct or the incorrect post. The percentage shown in the cell is simply how many of those 100 outcomes favoured the correct post, divided by 100.

In [ ]:
import e1_utils.e1_analysis_optimized as e1
importlib.reload(e1)

from e1_utils.e1_analysis_optimized import plot_ab_grid

# Likes only
#plot_ab_grid(OUTPUT_DIR, "e1_results_likes_only_paired.json",
#             title="A/B Like Decision — Likes Only (Correct % by Reaction Scale)")

plot_ab_grid(OUTPUT_DIR, "e1_results_likes_only_noise_paired.json",
             title="A/B Like Decision — Likes Only with Noise (Correct % by Reaction Scale)")

# Metrics
plot_ab_grid(OUTPUT_DIR, "e1_results_metrics_paired.json", 
             title="A/B Like Decision — Metrics (Correct % by Reaction Scale)")



In [ ]:
from e1_utils.e1_analysis_optimized import plot_ab_grid, plot_ab_diff_grid


# Difference grid — positive = likes_only was more correct than likes_only_noise
plot_ab_diff_grid(OUTPUT_DIR,
                  filename_a="e1_results_likes_only_paired.json",
                  filename_b="e1_results_likes_only_noise_paired.json",
                  title="Δ A/B Like Decision — Likes Only vs Likes Only with Noise")

**Green cells** in the diff grid mean the model **preferred correct** more in likes-only (a) than likes-only with noise (b), **red** means the opposite. 

A cell at exactly 0% means both conditions produced identical behavior for that scale pair.

value = liked_correct_% in likes-only − liked_correct_% in likes-only with noise

- Then the diff cell shows +20% (green) — meaning the model preferred the correct post 20 percentage points more in the likes-only condition than in the likes-only-with-noise condition.
  
- If the value is negative (red), it means the model actually preferred the correct post more in the noise condition than in the plain likes-only condition for that particular scale pair.
  
- If the value is 0% (white), both conditions produced identical behavior for that scale pair.


In [ ]:

import e1_utils.e1_analysis_optimized as e1
importlib.reload(e1)

from e1_utils.e1_analysis_optimized import plot_cc_grid

# Correct vs. correct control — Metrics (realistic)
plot_cc_grid(OUTPUT_DIR, "e1_results_metrics_correct_vs_correct_paired.json",
             title="Correct vs. Correct — Metrics (Chose Higher Engagement %)")

# Correct vs. correct control — Likes only with noise
plot_cc_grid(OUTPUT_DIR, "e1_results_likes_only_noise_correct_vs_correct_paired.json",
             title="Correct vs. Correct — Likes Only with Noise (Chose Higher Engagement %)")


## Two-step prompt pilot (supervisor item, 2026-08)

Tests whether making the model verify factual correctness explicitly *before* the popularity-influenced like/scroll decision improves the diagonal (competence, disparity=0). Two designs run head-to-head, diagonal cells only (7 scales x 100 images = 700 trials each, not the full 49-cell grid -- this is specifically about the diagonal, see `experiments/e1/e1_utils/e1_two_step.py` module docstring for the full rationale and why the grid is narrowed here). Compare each design's `e1_results_metrics_diagonal_twostep_*.json` diagonal accuracy against this notebook's existing single-step `e1_results_metrics_paired.json` diagonal accuracy (Section further up).

Currently pointed at the same `metrics_simple_plot_bigfont` images already used elsewhere in this notebook. If/when the font-size-adjusted ("bigfont") images are composited into full posts, swap `correct_base`/`incorrect_base` below for that directory instead.

In [ ]:
import time
from e1_utils.e1_two_step import run_e1_metrics_paired_twostep, DIAGONAL_PAIRS

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

start = time.time()
run_e1_metrics_paired_twostep(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
                               design="paired_verdict", output_filename="e1_results_metrics_diagonal_twostep_paired_verdict.json",
                               inference_fn=run_inference_qwen,
                               baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",
                               baseline_incorrect_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs")
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min ({elapsed / (len(selected_numbers) * len(DIAGONAL_PAIRS)):.2f}s/pair)")

In [ ]:
import time
from e1_utils.e1_two_step import run_e1_metrics_paired_twostep, DIAGONAL_PAIRS

correct_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/realistic"
incorrect_base = ROOT_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/incorrect/PNGs/realistic"
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

start = time.time()
run_e1_metrics_paired_twostep(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
                               design="isolated_verdict", output_filename="e1_results_metrics_diagonal_twostep_isolated_verdict.json",
                               inference_fn=run_inference_qwen,
                               baseline_correct_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/correct/PNGs",
                               baseline_incorrect_dir=ROOT_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont/incorrect/PNGs")
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min ({elapsed / (len(selected_numbers) * len(DIAGONAL_PAIRS)):.2f}s/pair)")

In [ ]:
import json

def diagonal_accuracy(path, filter_diagonal=False):
    records = json.loads((OUTPUT_DIR / path).read_text())
    if filter_diagonal:
        records = [r for r in records if r["correct_scale"] == r["incorrect_scale"]]
    return 100 * sum(r["liked_variant"] == "correct" for r in records) / len(records), len(records)

single_pct, single_n = diagonal_accuracy("e1_results_metrics_paired.json", filter_diagonal=True)
paired_pct, paired_n = diagonal_accuracy("e1_results_metrics_diagonal_twostep_paired_verdict.json")
isolated_pct, isolated_n = diagonal_accuracy("e1_results_metrics_diagonal_twostep_isolated_verdict.json")

print(f"Single-step (existing, diagonal cells only): {single_pct:.1f}%  (n={single_n})")
print(f"Two-step, paired_verdict:                    {paired_pct:.1f}%  (n={paired_n})")
print(f"Two-step, isolated_verdict:                  {isolated_pct:.1f}%  (n={isolated_n})")